In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import unicodedata
import difflib
try:
    from babel import Locale
    import pycountry_convert as pc
except ImportError:
    Locale = None
    pc = None
    print("Warning: 'babel'/'pycountry_convert' non installati. I continents saranno 'Altro'. "
          "Installa con: pip install babel pycountry_convert")

FILE_IHME = "ridotto.csv"

df_raw = pd.read_csv(FILE_IHME)

COL_LOCATION = "location_name"
COL_SEX = "sex_name"
COL_YEAR = "year"
COL_VALUE = "val"

etichette_sesso = df_raw[COL_SEX].unique()
etichetta_maschi = next(e for e in etichette_sesso
                         if "masch" in str(e).lower() or "male" in str(e).lower())
etichetta_femmine = next(e for e in etichette_sesso
                          if "femmin" in str(e).lower() or "female" in str(e).lower())

pivot = df_raw.pivot_table(
    index=[COL_LOCATION, COL_YEAR],
    columns=COL_SEX,
    values=COL_VALUE,
    aggfunc="first",
).reset_index()

pivot = pivot.rename(columns={etichetta_maschi: "share_men", etichetta_femmine: "share_women"})
pivot = pivot.dropna(subset=["share_men", "share_women"])

PAESI_ESCLUSI = ["Stati Uniti d'America"]

annotazioni_outlier = {}
for paese in PAESI_ESCLUSI:
    dati_paese = pivot[pivot[COL_LOCATION] == paese].set_index(COL_YEAR)
    for anno, riga in dati_paese.iterrows():
        testo_riga = f"{paese} (fuori scala): donne {riga['share_women']*100:.1f}%, uomini {riga['share_men']*100:.1f}%"
        annotazioni_outlier[int(anno)] = annotazioni_outlier.get(int(anno), "") + testo_riga + "  "

pivot = pivot[~pivot[COL_LOCATION].isin(PAESI_ESCLUSI)]

ECCEZIONI_MANUALI = {
    "Venezuela (Repubblica Bolivariana del)": "VE",
    "Bolivia (Stato Plurinazionale della)": "BO",
    "Repubblica Democratica Popolare del Laos": "LA",
    "Iran (Repubblica Islamica dell')": "IR",
}


def normalizza(s):
    s = s.replace("’", "'").strip()
    return unicodedata.normalize("NFKD", s).lower()


def costruisci_mappa_nomi():
    if Locale is None:
        return {}
    territori = Locale("it").territories  
    return {normalizza(nome): iso2 for iso2, nome in territori.items() if len(iso2) == 2}


_NOME_TO_ISO2 = costruisci_mappa_nomi()
_NOMI_NORMALIZZATI = list(_NOME_TO_ISO2.keys())

_MAPPING_CONTINENTE = {
    "NA": "North America", "SA": "South America", "AF": "Africa",
    "EU": "Europe", "AS": "Asia", "OC": "Oceania",
}


def get_continent(nome_paese):
    if Locale is None or pc is None:
        return "Altro"
    if nome_paese in ECCEZIONI_MANUALI:
        iso2 = ECCEZIONI_MANUALI[nome_paese]
    else:
        chiave = normalizza(nome_paese)
        iso2 = _NOME_TO_ISO2.get(chiave)
        if iso2 is None:
            corrispondenza = difflib.get_close_matches(chiave, _NOMI_NORMALIZZATI, n=1, cutoff=0.6)
            iso2 = _NOME_TO_ISO2.get(corrispondenza[0]) if corrispondenza else None
    if iso2 is None:
        return "Altro"
    try:
        code = pc.country_alpha2_to_continent_code(iso2)
        return _MAPPING_CONTINENTE.get(code, "Altro")
    except Exception:
        return "Altro"


pivot["Continent"] = pivot[COL_LOCATION].apply(get_continent)

NOMI_BREVI = {
    "Africa Subsahariana Meridionale": "Sud-Africa",
    "Africa Subsahariana Orientale": "Est-Africa",
    "Africa Subsahariana Occidentale": "Ovest-Africa",
    "Africa Subsahariana Centrale": "Africa Centrale",
    "Europa Orientale": "Est-EU",
    "Iran (Repubblica Islamica dell')": "Iran",
    "Repubblica di Corea": "Sud-Corea",
}

pivot["label_text"] = pivot[COL_LOCATION].apply(lambda nome: NOMI_BREVI.get(nome, nome))

color_map = {
    "North America": "#e0765a",
    "South America": "#8c3b45",
    "Africa": "#AA69A4",
    "Europe": "#5d7ea8",
    "Asia": "#2c7c6c",
    "Oceania": "#4fb3c4",
    "Altro": "#bbbbbb",
    "NotaOutlier": "rgba(0,0,0,0)",
}

x_max = pivot["share_women"].max() * 1.15
y_max = pivot["share_men"].max() * 1.15

righe_nota = []
for anno in pivot[COL_YEAR].unique():
    righe_nota.append({
        COL_LOCATION: "NotaOutlier",
        COL_YEAR: anno,
        "share_women": x_max * 0.97,
        "share_men": y_max * 0.04,
        "Continent": "NotaOutlier",
        "label_text": annotazioni_outlier.get(int(anno), ""),
    })
pivot_con_nota = pd.concat([pivot, pd.DataFrame(righe_nota)], ignore_index=True)

fig = px.scatter(
    pivot_con_nota.sort_values(COL_YEAR),
    x="share_women",
    y="share_men",
    animation_frame=COL_YEAR,
    animation_group=COL_LOCATION,
    color="Continent",
    hover_name=COL_LOCATION,
    text="label_text",
    color_discrete_map=color_map,
    range_x=[0, x_max],
    range_y=[0, y_max],
)

fig.update_traces(marker=dict(size=10), textposition="top center", textfont_size=8)
fig.update_traces(
    selector=dict(name="NotaOutlier"),
    marker=dict(size=0, opacity=0),
    textposition="middle left",
    textfont=dict(size=11, color="#888888"),
    hoverinfo="skip",
    hovertemplate=None,
    showlegend=False,
)

fig.update_layout(
    xaxis_title="Donne",
    yaxis_title="Uomini",
    xaxis_tickformat=".1%",
    xaxis_nticks=5,
    yaxis_tickformat=".1%",
    template="simple_white",
    legend_title_text="",
)

fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 500
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["redraw"] = True

for step in fig.layout.sliders[0].steps:
    step.args[1]["frame"]["redraw"] = True

fig.show()